## Import Required Libraries

In [1]:
# ==========================================================
# Import Required Libraries
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from pathlib import Path

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.float_format", "{:.2f}".format)

## Load Merged Dataset

In [2]:
# Load merged customer dataset created during the data integration step

# Define data directory
INTERIM_DIR = Path("../data/interim")

# Load merged customer dataset
cust_data = pd.read_csv(
    INTERIM_DIR / "01_merged_customer_data.csv"
)

cust_data.head()

,ID,Age,CustomerSince,HighestSpend,ZipCode,HiddenScore,MonthlyAverageSpend,Level,Mortgage,Security,FixedDepositAccount,InternetBanking,CreditCard,LoanOnCard
0,1,25,1,49,91107,4,1.60,1,0,1,0,0,0,NaN
1,2,45,19,34,90089,3,1.50,1,0,1,0,0,0,NaN
2,3,39,15,11,94720,1,1.00,1,0,0,0,0,0,NaN
3,4,35,9,100,94112,1,2.70,2,0,0,0,0,0,NaN
4,5,35,8,45,91330,4,1.00,2,0,0,0,0,1,NaN


## Dataset Overview

In [3]:
# Dataset Overview

print(f"Rows    : {cust_data.shape[0]}")
print(f"Columns : {cust_data.shape[1]}")

Rows    : 5000
Columns : 14


## Recreate Feature Groups

In [4]:
# ==========================================================
# Feature Classification
# ==========================================================

target_col = "LoanOnCard"

id_cols = ["ID"]

location_cols = ["ZipCode"]

num_cols = [
    "Age",
    "CustomerSince",
    "HighestSpend",
    "MonthlyAverageSpend",
    "Mortgage"
]

ordinal_cols = [
    "Level",
    "HiddenScore"
]

binary_cols = [
    "Security",
    "FixedDepositAccount",
    "InternetBanking",
    "CreditCard"
]

## Missing Value Treatment

In [5]:
# Missing Value Assessment

cust_data.isnull().sum()

ID                      0
Age                     0
CustomerSince           0
HighestSpend            0
ZipCode                 0
HiddenScore             0
MonthlyAverageSpend     0
Level                   0
Mortgage                0
Security                0
FixedDepositAccount     0
InternetBanking         0
CreditCard              0
LoanOnCard             20
dtype: int64

In [6]:
# Remove Missing Target Observations

cust_data = cust_data.dropna(
    subset=[target_col]
)

cust_data.shape

(4980, 14)

## Explanation
Only the target variable (LoanOnCard) contained missing values.
Since the target variable represents the outcome to be predicted,
records with missing target values cannot contribute to supervised
learning and were therefore removed from the dataset.

## Convert Target Variable to Integer

In [7]:
# LoanOnCard is a binary target variable (0 = No Loan, 1 = Loan)

cust_data["LoanOnCard"] = cust_data["LoanOnCard"].astype(int)

print(cust_data["LoanOnCard"].unique())
print(cust_data["LoanOnCard"].dtype)

[1 0]
int32


## Since CSV does not preserve pandas categorical dtypes, mention that categorical treatment will be applied again during modeling.

## Duplicate Validation

In [8]:
# Duplicate Record Check

cust_data.duplicated().sum()

0

## Investigate CustomerSince

In [9]:
cust_data["CustomerSince"].value_counts().sort_index()

CustomerSince
-3       4
-2      15
-1      32
 0      66
 1      73
 2      84
 3     129
 4     113
 5     146
 6     119
 7     121
 8     118
 9     145
 10    117
 11    116
 12    102
 13    116
 14    127
 15    118
 16    125
 17    125
 18    137
 19    134
 20    148
 21    113
 22    121
 23    144
 24    130
 25    142
 26    133
 27    124
 28    138
 29    124
 30    126
 31    104
 32    154
 33    117
 34    125
 35    143
 36    113
 37    116
 38     88
 39     85
 40     57
 41     42
 42      8
 43      3
Name: count, dtype: int64

### Note: Total = 51 records (earlier check showed 52; likely one row was removed while dropping missing targets)

## Observations

The CustomerSince feature contains a small number of negative values:
- -3 : 4 records
- -2 : 15 records
- -1 : 32 records

representing approximately 1% of the dataset.

### Interpretation

Since the variable is masked by the bank and the negative values form a natural continuation of the overall distribution, there is insufficient evidence to classify them as erroneous observations.

### Action Taken
- No modification was performed.
- The negative values were retained for subsequent analysis.

### Rationale

Removing or altering these observations may result in the loss of potentially useful information and could introduce unintended bias into the dataset.

## Data Type Optimization

In [10]:
# Convert ordinal variables.

cust_data[ordinal_cols] = (
    cust_data[ordinal_cols]
    .astype("category")
)

In [11]:
# Convert binary variables.

cust_data[binary_cols] = (
    cust_data[binary_cols]
    .astype("int8")
)

In [12]:
# Final Data Quality Validation

print("Missing Values:")
print(cust_data.isnull().sum().sum())

print("\nDuplicate Records:")
print(cust_data.duplicated().sum())

Missing Values:
0

Duplicate Records:
0


## Save Cleaned Dataset

In [13]:
# Save Cleaned Customer Dataset

output_file = Path("../data/interim/02_cleaned_customer_data.csv")
output_file.parent.mkdir(parents=True, exist_ok=True)

cust_data.to_csv(
    output_file,
    index=False
)

print(f"Cleaned dataset saved successfully at: {output_file}")

Cleaned dataset saved successfully at: ..\data\interim\02_cleaned_customer_data.csv


## Phase 2: Data Cleaning & Preprocessing Summary
### Overview

The objective of this phase was to address data quality issues identified during the Data Understanding phase and prepare the dataset for exploratory data analysis and feature engineering.

The cleaning process focused on handling missing values, validating duplicate records, investigating anomalous observations, optimizing data types, and ensuring overall dataset consistency.

### Missing Value Treatment

A review of missing values revealed that only the target variable (LoanOnCard) contained missing observations.

| Feature | Missing Values |
|----------|---------------|
| LoanOnCard | 20 |

Since the target variable is required for supervised learning, records with missing target values were removed from the dataset.

#### Action Taken
Removed 20 observations with missing LoanOnCard values.

#### Result
| Metric | Before | After |
|---------|--------|-------|
| Records | 5,000 | 4,980 |

---

### Duplicate Record Validation

Duplicate analysis confirmed that:
- No duplicate records were present.
- No duplicate customer IDs were identified.

#### Action Taken
- No action required.

---

### CustomerSince Investigation

The CustomerSince feature contained a small number of negative values.

| Value | Count |
|--------|-------|
| -3 | 4 |
| -2 | 15 |
| -1 | 32 |

These observations represented approximately 1% of the dataset.

#### Investigation Findings
- The values form a natural continuation of the overall distribution.
- No extreme negative values were observed.
- The variable is masked by the bank, making direct interpretation impossible.

#### Action Taken
- Negative CustomerSince values were retained.

#### Rationale

There was insufficient evidence to classify these observations as data quality errors. Removing or modifying them could potentially eliminate useful predictive information.

---

### Data Type Optimization

To improve memory efficiency and prepare the dataset for analysis:

#### Ordinal Variables

The following variables were converted to categorical data types:
- Level
- HiddenScore
- Binary Variables

The following variables were converted to compact integer types:
- Security
- FixedDepositAccount
- InternetBanking
- CreditCard

---

### Final Data Quality Validation

The dataset was re-evaluated after cleaning.

Results
| Validation Check | Result |
|------------------|--------|
| Missing Values | 0 |
| Duplicate Records | 0 |
| Invalid Age Values | 0 |
| Negative Spending Values | 0 |
| Negative Mortgage Values | 0 |

The dataset now contains complete target information and no identified data integrity issues requiring further cleaning.

Cleaned Dataset Summary
| Metric | Value |
|---------|-------|
| Total Records | 4,980 |
| Total Features | 14 |
| Missing Values | 0 |
| Duplicate Records | 0 |

---

### Conclusion

The dataset was successfully cleaned and prepared for exploratory data analysis. Missing target observations were removed, data consistency was validated, and feature data types were optimized. No major data quality issues remain.

The cleaned dataset is now suitable for detailed exploratory analysis, statistical testing, feature engineering, and machine learning model development.

## Phase 2 Deliverables Completed

- Missing Value Treatment
- Duplicate Validation
- CustomerSince Investigation
- Data Type Optimization
- Final Data Quality Validation
- Cleaned Dataset Export
- Data Cleaning & Preprocessing Summary